In [ ]:
# Load or reload R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    try:
        %reload_ext rpy2.ipython
    except Exception as e2:
        print("Note on rpy2 initialization:", e2)

# Master Triage Pipeline: 3-Layer Hierarchical LightGBM Benchmark (`models/combined_hierarchical_triage_pipeline.ipynb`)

This notebook evaluates the **3-Layer Hierarchical LightGBM Triage Pipeline** comparing two routing algorithms on the **1% Holdout Test Set**:

### 3-Layer Hierarchical LightGBM Architecture
- **Layer 1 LightGBM (ESI 1 Detector)**: Trained on **complete data** (`lightgbm_layer1_esi1_model.rds`).
- **Layer 2 LightGBM (ESI 2/3 vs ESI 4/5 Specialist)**: Trained **without ESI 1 rows** (`rf_esi23_esi45_extreme_model.rds`).
- **Layer 3A LightGBM (ESI 2 vs ESI 3 Specialist)**: Trained strictly on ESI 2 & ESI 3 rows (`lightgbm_esi23_model.rds`).
- **Layer 3B LightGBM (ESI 4 vs ESI 5 Specialist)**: Trained strictly on ESI 4 & ESI 5 rows (`lightgbm_esi45_model.rds`).

### Evaluated Routing Algorithms
1. **Probabilistic Joint Product Algorithm (Soft Scaling)**:
   - Computes continuous joint probability products across all 3 layers:
     - $P(\text{ESI 1}) = P_{L1}(\text{ESI 1})$
     - $P(\text{ESI 2}) = (1 - P_{L1}(\text{ESI 1})) \times P_{L2}(\text{ESI 2\_3}) \times P_{L3A}(\text{ESI 2})$
     - $P(\text{ESI 3}) = (1 - P_{L1}(\text{ESI 1})) \times P_{L2}(\text{ESI 2\_3}) \times (1 - P_{L3A}(\text{ESI 2}))$
     - $P(\text{ESI 4}) = (1 - P_{L1}(\text{ESI 1})) \times (1 - P_{L2}(\text{ESI 2\_3})) \times P_{L3B}(\text{ESI 4})$
     - $P(\text{ESI 5}) = (1 - P_{L1}(\text{ESI 1})) \times (1 - P_{L2}(\text{ESI 2\_3})) \times (1 - P_{L3B}(\text{ESI 4}))$
2. **Hard Case-Selector Routing Algorithm (If-Else Routing)**:
   - Uses Layer 1 as a hard detector (`if L1 >= 0.5 -> ESI 1; else if L2 >= 0.5 -> L3A (ESI 2/3); else -> L3B (ESI 4/5)`).

### Evaluated Metrics
Evaluates **Recall (Sensitivity)**, **Specificity**, **Balanced Accuracy**, and **ROC-AUC** across both algorithms.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(lightgbm)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last"); p_max    <- get_vec("pulse_max"); p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last");   s_max    <- get_vec("sbp_max");   s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last");  o2_max   <- get_vec("spo2_max");  o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last");   r_max    <- get_vec("resp_max");   r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
df_full <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_last              = p_last,
  resp_last               = r_last,
  spo2_last               = o2_last,
  sbp_last                = s_last,
  pulse_min               = p_min,
  resp_min                = r_min,
  spo2_min                = o2_min,
  sbp_min                 = s_min,
  pulse_max               = p_max,
  resp_max                = r_max,
  spo2_max                = o2_max,
  sbp_max                 = s_max,
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_full <- na.omit(df_full)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_col, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
cat(sprintf("Data Partition Summary: Train=%d, Val=%d, Test=%d\n", nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Trained Sub-Model Artifacts from deploy/
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
lgb_l1_file <- file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds")
lgb_l2_file <- file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds")
lgb_esi23_file <- file.path(deploy_dir, "lightgbm_esi23_model.rds")
lgb_esi45_file <- file.path(deploy_dir, "lightgbm_esi45_model.rds")
lgb_l1_obj <- readRDS(lgb_l1_file)
l1_model   <- if (is.list(lgb_l1_obj) && "model" %in% names(lgb_l1_obj)) lgb_l1_obj$model else lgb_l1_obj
lgb_l2_obj <- readRDS(lgb_l2_file)
l2_model   <- if (is.list(lgb_l2_obj) && "model" %in% names(lgb_l2_obj)) lgb_l2_obj$model else lgb_l2_obj
lgb23_obj <- readRDS(lgb_esi23_file)
lgb23_model <- if (is.list(lgb23_obj) && "model" %in% names(lgb23_obj)) lgb23_obj$model else lgb23_obj
lgb45_obj <- readRDS(lgb_esi45_file)
lgb45_model <- if (is.list(lgb45_obj) && "model" %in% names(lgb45_obj)) lgb45_obj$model else lgb45_obj
preproc <- if (is.list(lgb_l1_obj) && "preproc" %in% names(lgb_l1_obj)) lgb_l1_obj$preproc else NULL
if (is.null(preproc)) {
  binary_cols <- c("gender", "cc_breathingdifficulty")
  cont_cols   <- setdiff(names(train_df), c(binary_cols, "target_col"))
  preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
}
test_scaled <- predict(preproc, test_df)
cat("All 4 LightGBM sub-models successfully loaded!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Run Inference & Compute Routing Decisions
# ---------------------------------------------------------
feat_names <- setdiff(names(test_scaled), "target_col")
X_test_mat <- as.matrix(test_scaled[, feat_names])
# Layer 1 Inference (ESI 1 Detector)
p_l1_esi1 <- predict(l1_model, X_test_mat)
# Layer 2 Inference (ESI 2/3 vs 4/5 Specialist)
p_l2_esi23 <- predict(l2_model, X_test_mat)
# Layer 3A (ESI 2 vs 3 Specialist) Inference
p_l3a_esi2 <- predict(lgb23_model, X_test_mat)
if (is.matrix(p_l3a_esi2)) p_l3a_esi2 <- p_l3a_esi2[, 2]
# Layer 3B (ESI 4 vs 5 Specialist) Inference
p_l3b_esi4 <- predict(lgb45_model, X_test_mat)
if (is.matrix(p_l3b_esi4)) p_l3b_esi4 <- p_l3b_esi4[, 2]
# ---------------------------------------------------------
# 1. Soft Probabilistic Joint Product Scaling
# ---------------------------------------------------------
p_non1 <- 1.0 - p_l1_esi1
probs_soft <- matrix(0, nrow = nrow(test_df), ncol = 5)
colnames(probs_soft) <- c("1", "2", "3", "4", "5")
probs_soft[, 1] <- p_l1_esi1
probs_soft[, 2] <- p_non1 * p_l2_esi23 * p_l3a_esi2
probs_soft[, 3] <- p_non1 * p_l2_esi23 * (1.0 - p_l3a_esi2)
probs_soft[, 4] <- p_non1 * (1.0 - p_l2_esi23) * p_l3b_esi4
probs_soft[, 5] <- p_non1 * (1.0 - p_l2_esi23) * (1.0 - p_l3b_esi4)
preds_soft <- factor(apply(probs_soft, 1, which.max), levels = 1:5)
# ---------------------------------------------------------
# 2. Hard Case-Selector Routing (If-Else)
# ---------------------------------------------------------
preds_hard <- numeric(nrow(test_df))
for (i in 1:nrow(test_df)) {
  if (p_l1_esi1[i] >= 0.5) {
    preds_hard[i] <- 1
  } else if (p_l2_esi23[i] >= 0.5) {
    preds_hard[i] <- ifelse(p_l3a_esi2[i] >= 0.5, 2, 3)
  } else {
    preds_hard[i] <- ifelse(p_l3b_esi4[i] >= 0.5, 4, 5)
  }
}
preds_hard <- factor(preds_hard, levels = 1:5)
act_test <- factor(as.numeric(as.character(test_df$target_col)), levels = 1:5)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Benchmark Metrics Comparison (Recall, Specificity, BalAcc, ROC-AUC ONLY)
# ---------------------------------------------------------
eval_algorithm <- function(preds, probs, alg_name) {
  cm <- confusionMatrix(preds, act_test)
  
  rec_list  <- as.numeric(cm$byClass[, "Sensitivity"])
  spec_list <- as.numeric(cm$byClass[, "Specificity"])
  bal_list  <- as.numeric(cm$byClass[, "Balanced Accuracy"])
  
  rec_list[is.na(rec_list)]   <- 0
  spec_list[is.na(spec_list)] <- 0
  bal_list[is.na(bal_list)]   <- 0
  
  auc_list <- sapply(1:5, function(i) {
    act_bin <- ifelse(act_test == i, 1, 0)
    if (!is.null(probs)) {
      p_col <- probs[, i]
    } else {
      p_col <- ifelse(preds == i, 1, 0)
    }
    r_obj <- tryCatch(pROC::roc(act_bin, p_col), error = function(e) NULL)
    if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
  })
  
  macro_rec  <- mean(rec_list)
  macro_spec <- mean(spec_list)
  macro_bal  <- mean(bal_list)
  macro_auc  <- mean(auc_list, na.rm = TRUE)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   5-CLASS BENCHMARK: %s\n", toupper(alg_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Macro Recall (Sens)     : %.4f\n", macro_rec))
  cat(sprintf("  Macro Specificity       : %.4f\n", macro_spec))
  cat(sprintf("  Macro Balanced Accuracy : %.4f\n", macro_bal))
  cat(sprintf("  Macro ROC-AUC           : %.4f\n", macro_auc))
  cat(sprintf("============================================================\n\n"))
  print(cm$table)
  cat("\n\n")
  
  return(data.frame(
    Algorithm = alg_name,
    Macro_Recall = round(macro_rec, 4),
    Macro_Specificity = round(macro_spec, 4),
    Macro_Balanced_Accuracy = round(macro_bal, 4),
    Macro_ROC_AUC = round(macro_auc, 4)
  ))
}
soft_res <- eval_algorithm(preds_soft, probs_soft, "Probabilistic_Joint_Product")
hard_res <- eval_algorithm(preds_hard, NULL,       "Hard_Case_Selector_IfElse")
comp_df <- rbind(soft_res, hard_res)
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
write.csv(comp_df, file = file.path(reports_dir, "combined_pipeline_test_report.csv"), row.names = FALSE)
cat("Benchmark Comparison Report written to reports/combined_pipeline_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Grouped Bar Chart Visualization
# ---------------------------------------------------------
df_long <- comp_df %>%
  pivot_longer(cols = c("Macro_Recall", "Macro_Specificity", "Macro_Balanced_Accuracy", "Macro_ROC_AUC"),
               names_to = "Metric", values_to = "Score") %>%
  mutate(Metric = factor(Metric, levels = c("Macro_Recall", "Macro_Specificity", "Macro_Balanced_Accuracy", "Macro_ROC_AUC")))
p <- ggplot(df_long, aes(x = Metric, y = Score, fill = Algorithm)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.8), width = 0.7) +
  geom_text(aes(label = sprintf("%.4f", Score)), position = position_dodge(width = 0.8), vjust = -0.4, size = 3.5) +
  scale_y_continuous(limits = c(0, 1.05), breaks = seq(0, 1, 0.1)) +
  scale_fill_manual(values = c("Probabilistic_Joint_Product" = "#1f77b4", "Hard_Case_Selector_IfElse" = "#ff7f0e")) +
  labs(title = "Master Triage Pipeline: Soft Probabilistic Joint vs Hard Case-Selector",
       subtitle = "Holdout Test Set Performance across Macro Metrics",
       x = "Metric", y = "Score", fill = "Algorithm") +
  theme_minimal(base_size = 13) +
  theme(legend.position = "top", plot.title = element_text(face = "bold", hjust = 0.5), plot.subtitle = element_text(hjust = 0.5))
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
ggsave(filename = file.path(plots_dir, "combined_pipeline_metrics_barchart.png"), plot = p, width = 9, height = 5.5, dpi = 300)
cat("Bar chart saved to plots/combined_pipeline_metrics_barchart.png\n")